In [3]:
import json
import cv2 as cv
from pathlib import Path

# Paths
base_dir = Path.cwd()  # current working directory
data_dir = base_dir / "labeled_example_data"
labels_path = data_dir / "labels.json"
output_dir = base_dir / "labeled_with_red_dot"
output_dir.mkdir(exist_ok=True)

# Load labels
with open(labels_path, "r") as f:
    labels = json.load(f)

# Loop through all entries
for entry in labels:
    img_path = data_dir / entry["filename"]
    img = cv.imread(str(img_path))
    if img is None:
        print(f"⚠️ Could not read {img_path}")
        continue

    h, w = img.shape[:2]

    # Draw red dot if wheel is present
    if entry["is_present"] == 1.0:
        x_px = int(entry["center"][0] * w)
        y_px = int(entry["center"][1] * h)
        cv.circle(img, (x_px, y_px), 5, (0, 0, 255), -1)  # red filled circle

    # Save annotated image
    out_path = output_dir / entry["filename"]
    cv.imwrite(str(out_path), img)

print(f"✅ Annotated images saved to: {output_dir}")

✅ Annotated images saved to: /Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/labeled_with_red_dot


In [7]:
import json
import cv2 as cv
import numpy as np
from pathlib import Path

# --- Paths ---
base_dir = Path.cwd()  # run from project root
data_dir = base_dir / "labeled_example_data"
labels_path = data_dir / "labels.json"
output_dir = base_dir / "debug_check"
output_dir.mkdir(exist_ok=True)

# --- Load labels ---
with open(labels_path, "r") as f:
    labels = json.load(f)
labels_by_name = {e["filename"]: e for e in labels}

# --- Ellipse fitting helper ---
def fit_best_ellipse_center(img):
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    gray = cv.equalizeHist(gray)
    edges = cv.Canny(gray, 50, 150)
    cnts, _ = cv.findContours(edges, cv.RETR_LIST, cv.CHAIN_APPROX_NONE)
    best, score = None, -1
    H, W = gray.shape[:2]
    for c in cnts:
        if len(c) < 60:
            continue
        try:
            ell = cv.fitEllipse(c)
        except cv.error:
            continue
        (cx, cy), (MA, ma), _ = ell
        if not (0 <= cx < W and 0 <= cy < H):
            continue
        a, b = max(MA, ma), min(MA, ma)
        ar = b / (a + 1e-6)  # aspect ratio
        area = np.pi * 0.25 * a * b
        s = 0.6 * ar + 0.4 * (area / (H * W))
        if s > score:
            best, score = (cx, cy), s
    return best

# --- Crop helper ---
def crop_around_center(img, cx, cy, box_size=300):
    H, W = img.shape[:2]
    half = box_size // 2
    x0 = max(0, cx - half)
    y0 = max(0, cy - half)
    x1 = min(W, cx + half)
    y1 = min(H, cy + half)
    return img[y0:y1, x0:x1], x0, y0

# --- Loop over all images ---
errors = []
fail_count = 0

for fname, entry in labels_by_name.items():
    img_path = data_dir / fname
    img = cv.imread(str(img_path))
    if img is None:
        print(f"⚠️ Missing image: {fname}")
        continue

    H, W = img.shape[:2]

    if entry["is_present"] != 1.0:
        # No wheel present — skip detection
        continue

    # Ground truth pixel coords
    gt_x = int(round(entry["center"][0] * W))
    gt_y = int(round(entry["center"][1] * H))

    # Crop around GT
    crop_img, x0, y0 = crop_around_center(img, gt_x, gt_y, box_size=300)

    # Detect ellipse center in crop
    geo_center_crop = fit_best_ellipse_center(crop_img)
    if geo_center_crop:
        gx_crop, gy_crop = map(int, map(round, geo_center_crop))
        gx_full = gx_crop + x0
        gy_full = gy_crop + y0

        # Compute pixel error
        error = np.hypot(gx_full - gt_x, gy_full - gt_y)
        errors.append(error)

        # Draw GT (red) and detected (green)
        cv.circle(img, (gt_x, gt_y), 6, (0, 0, 255), -1)
        cv.drawMarker(img, (gt_x, gt_y), (0, 0, 255),
                      markerType=cv.MARKER_CROSS, markerSize=16, thickness=2)
        cv.circle(img, (gx_full, gy_full), 6, (0, 255, 0), 2)
        cv.drawMarker(img, (gx_full, gy_full), (0, 255, 0),
                      markerType=cv.MARKER_TILTED_CROSS, markerSize=20, thickness=2)
    else:
        fail_count += 1
        # Draw only GT
        cv.circle(img, (gt_x, gt_y), 6, (0, 0, 255), -1)
        cv.drawMarker(img, (gt_x, gt_y), (0, 0, 255),
                      markerType=cv.MARKER_CROSS, markerSize=16, thickness=2)

    # Save overlay
    out_path = output_dir / fname
    cv.imwrite(str(out_path), img)

# --- Summary ---
if errors:
    avg_error = sum(errors) / len(errors)
    print(f"✅ Processed {len(errors)} images with detections")
    print(f"📏 Average pixel error: {avg_error:.2f}")
print(f"⚠️ Failed detections: {fail_count}")
print(f"Overlays saved to: {output_dir}")

✅ Processed 1000 images with detections
📏 Average pixel error: 31.72
⚠️ Failed detections: 0
Overlays saved to: /Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/debug_check
